In [1]:
import os, sys
import datetime
import csv
import pandas as pd
import numpy as np
import argparse
from tqdm import tqdm
# from zUtils import zData

import ipywidgets as widgets
from ipywidgets import interact, interact_manual

import django
#from djCOADD import djOrgDB


In [2]:
djConfig = 'Laptop'

# Django Folder -------------------------------------------------------------
if djConfig == 'Meran':
    djDir = "D:/Code/zdjCode/adjCOADD"
#   uploadDir = "C:/Code/A02_WorkDB/03_Django/adjCOADD/utilities/upload_data/Data"
#   orgdbDir = "C:/Users/uqjzuegg/The University of Queensland/IMB CO-ADD - OrgDB"
elif djConfig == 'Linux':
    djDir = "/home/uqjzuegg/Code/zdjCode/adjCOADD"
#     uploadDir = "C:/Data/A02_WorkDB/03_Django/adjCOADD/utilities/upload_data/Data"
elif djConfig == 'Laptop':
    djDir = "C:/Code/zdjCode/adjCOADD"
#     uploadDir = "/home/uqjzuegg/DeepMicroB/Code/Python/Django/adjCOADD/utilities/upload_data/Data"
else:
    djDir = None

# Django -------------------------------------------------------------
sys.path.append(djDir)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "adjcoadd.settings")

# Needed for Jupyter NoteBook
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()
from apputil.models import ApplicationUser, Dictionary

print()
print(f"Python         : {sys.version.split('|')[0]}")
print(f"Conda Env      : {os.environ['CONDA_DEFAULT_ENV']}")
#logger.info(f"LogFile        : {logFileName}")

print(f"Django         : {django.__version__}")
print(f"Django Folder  : {djDir}")
print(f"Django Project : {os.environ['DJANGO_SETTINGS_MODULE']}")

Project: adjCOADD 
BaseDir: C:\Code\zdjCode\adjCOADD
Version: 1.4.0
Host Name: imb-coadd-db.imb.uq.edu.au

Python         : 3.11.4 
Conda Env      : dj42py311
Django         : 4.2.2
Django Folder  : C:/Code/zdjCode/adjCOADD
Django Project : adjcoadd.settings


In [ ]:
from dplate.models import TestPlate
RunId = 'HF_UQ39_R01'
PlateId = 'E00100049'
 
TestPlates = {}
TestPlates[PlateId] = TestPlate.get(PlateId,WellData=True)

for key in TestPlates:
    TestPlates[key].apply_layout()
    TestPlates[key].calc_inhibition()
    TestPlates[key].make_wells_df(RowCol=True, ListToString=True, ReadoutField=True)
TestPlates_Names = list(TestPlates.keys())
TestPlates_Names.sort()  

#['COMPOUNDS','SETS','CONCS','CONC_UNITS','READOUT','INHIBITION','ISNEGCONTROL','ISPOSCONTROL','ISCONTROL','ISSKIP']
# print(TestPlates[PlateId].zfactor)
# print(TestPlates[PlateId].plate_quality)

0.936
Valid


In [7]:
pd.set_option('display.max_columns', 24)
PivotBy_List = ['cmpbatches','concs','conc_units','is_negcontrol','is_poscontrol','is_control','is_sample']

@interact
def show_plate(Plate =TestPlates_Names, 
               PivotBy=PivotBy_List):
    #well_df = TestPlates[Plate].make_wells_df_df(RowCol=True)
    _welldata = TestPlates[Plate].wells_df.replace({False: '-', True: 'Y'})
    prop_map = _welldata.pivot_table(index="row", columns="col", values=PivotBy, aggfunc=lambda x: list(x)[0])
    print(f"{TestPlates[Plate]}: {TestPlates[Plate].assay_id} - {TestPlates[Plate].control_layout}")
    print(f"Statistics  Neg: {TestPlates[Plate].negcontrol_stats} ")
    print(f"            Pos: {TestPlates[Plate].poscontrol_stats} ")
    print(f"            Smp: {TestPlates[Plate].sample_stats}")
    print(f"            Plt: [{TestPlates[Plate].plate_quality}] Zf:{TestPlates[Plate].zfactor} ")
    display(prop_map)

interactive(children=(Dropdown(description='Plate', options=('E00100049',), value='E00100049'), Dropdown(descr…

In [8]:
@interact
def plot_plate(Plate=TestPlates_Names,
               Plot=['readout_1','inhibition']):
    TestPlates[Plate].plot_heatmap(Plot,None,propLegend=False)

interactive(children=(Dropdown(description='Plate', options=('E00100049',), value='E00100049'), Dropdown(descr…